# Explore: Amtrak Trains

## Intercity Passenger Rail Service Station Performance Metrics

The Amtrak [network](https://www.amtrak.com/content/dam/projects/dotcom/english/public/documents/Maps/Amtrak-System-Map-020923.pdf)
is a passenger rail service that provides intercity rail service in the
continental United States and to select Canadian cities. The network is operated by the
[National Railroad Passenger Corporation](https://railroads.dot.gov/passenger-rail/amtrak/amtrak),
a federally chartered for-profit corporation that receives some state funding and covers its
operating costs by selling tickets and providing other services.

This notebook commences exploration of the augmented quarterly
[Amtrak](https://www.amtrak.com/home.html) station performance metrics. The goal is to better
understand individual Amtrak train performance and identify potential areas for further analysis.

### Variable names

A number of variable names in this project leverage the following abbreviations. The naming
strategy is to strike a balance between brevity and readability:

* `amtk`: Amtrak (reporting mark)
* `chrt`: chart
* `cols`: columns
* `const`: constant
* `cwd`: current working directory
* `eb`: eastbound direction of travel
* `lm`: linear model
* `mi`: miles
* `mm`: minutes (ISO 8601)
* `nb`: northbound direction of travel
* `psgr`: passenger
* `qtr`: quarter
* `rte`: route
* `sb`: southbound direction of travel
* `stats`: summary statistics
* `stn`: station
* `stns`: stations
* `svc`: service
* `trn`: train
* `wb`: westbound direction of travel

In [1]:
import numpy as np
import pandas as pd
import pathlib as pl
import tomllib as tl

import fra_amtrak.amtk_detrain as detrn
import fra_amtrak.amtk_frame as frm
import fra_amtrak.amtk_network as ntwk
import fra_amtrak.chart.chart as chrt
import fra_amtrak.chart.builders.bxplt_preagg_bldr as bxp_bld
import fra_amtrak.chart.builders.hstgrm_bldr as hst_bld
import fra_amtrak.chart.schemas.bxplt_preagg as bxp
import fra_amtrak.chart.schemas.hstgrm as hst


In [120]:
import warnings, numpy as np, scipy.stats as stats
import fra_amtrak.amtk_detrain as detrn
import fra_amtrak.amtk_frame as frm
import fra_amtrak.amtk_network as ntwk
import pandas as pd

# ── Patch compute_sum_stats ──────────────────────────────────
def compute_sum_stats(frame, agg_columns, agg_funcs, precision=4):
    agg_dict = {col: agg_funcs for col in frame.columns if col in agg_columns}
    return frame.agg(agg_dict).round(precision)
detrn.compute_sum_stats = compute_sum_stats

# ── Patch compute_sum_stats_by_group ────────────────────────
def compute_sum_stats_by_group(frame, groups, agg_columns, agg_funcs, precision=4):
    agg_dict = {col: agg_funcs for col in frame.columns if col in agg_columns}
    return frame.groupby(groups).agg(agg_dict).round(precision)
detrn.compute_sum_stats_by_group = compute_sum_stats_by_group

# ── Patch describe_numeric_column ───────────────────────────
def describe_numeric_column(column):
    try:
        if np.issubdtype(column.dtype, np.number):
            min_ = column.min()
            max_ = column.max()
            return {
                "type": column.__class__, "name": column.name,
                "values": {"non_null": column.count(), "missing": column.isna().sum(), "dtype": column.dtype},
                "center": {"mean": column.mean(), "median": column.median(), "mode": column.mode()[0] if not column.mode().empty else None},
                "position": {"min": np.float64(min_), "25%": np.float64(column.quantile(0.25)), "50%": np.float64(column.quantile()), "75%": np.float64(column.quantile(0.75)), "max": np.float64(max_)},
                "spread": {"variance": column.var(), "std": column.std(), "range": max_ - min_, "iqr": stats.iqr(column)},
                "shape": {"skewness": column.skew(), "kurtosis": column.kurt()}
            }
    except Exception as e:
        warnings.warn(str(e), UserWarning)
    return None
frm.describe_numeric_column = describe_numeric_column

# ── Patch bin_data ───────────────────────────────────────────
def bin_data(frame, column, bins):
    binned_data = frame.copy()
    binned_data["bin"] = pd.cut(binned_data[column], bins=bins, labels=False, include_lowest=True)
    binned_data = binned_data.dropna(subset=["bin"]).copy()
    binned_data["bin"] = binned_data["bin"].astype(int)
    binned_data["bin_start"] = binned_data["bin"].apply(lambda x: bins[x])
    binned_data["bin_end"] = binned_data["bin"].apply(lambda x: bins[x + 1] if x + 1 < len(bins) else bins[x])
    binned_data["bin_center"] = (binned_data["bin_start"] + binned_data["bin_end"]) / 2
    return binned_data
frm.bin_data = bin_data

# ── Patch by_train_number ────────────────────────────────────
def by_train_number(frame, train_number):
    return frame[frame[COLS["trn"]] == train_number].reset_index(drop=True)
ntwk.by_train_number = by_train_number

# ── Patch by_sub_service ─────────────────────────────────────
def by_sub_service(frame, sub_service):
    return frame[frame[COLS["sub_svc"]] == sub_service].reset_index(drop=True)
ntwk.by_sub_service = by_sub_service

# ── Patch format_year_quarter ────────────────────────────────
def format_year_quarter(row):
    return f"{int(row[COLS['year']])}Q{int(row[COLS['quarter']])}"
detrn.format_year_quarter = format_year_quarter

# ── Patch assign_color ───────────────────────────────────────
def assign_color(fiscal_quarter, colors=None):
    if colors is None:
        colors = [COLORS["amtk_blue"], COLORS["amtk_red"]]
    return colors[0] if int(fiscal_quarter[-1]) % 2 == 0 else colors[1]
detrn.assign_color = assign_color

def by_sub_service(frame, sub_service):
    mask = frame[COLS["sub_svc"]].str.lower() == sub_service.lower()
    return frame[mask].reset_index(drop=True)
ntwk.by_sub_service = by_sub_service

print("✓ Semua patch berhasil!")

✓ Semua patch berhasil!


## 1.0 Read files

### 1.1 Resolve paths

In [9]:
parent_path = pl.Path.cwd()  # current working directory
parent_path

PosixPath('/home/jovyan/work/assignments/Course4')

### 1.2 Load constants

Load a companion [TOML](https://toml.io/en/) file named `notebook.toml` containing constants.

In [10]:
filepath = parent_path.joinpath("notebook.toml")
with open(filepath, "rb") as file_obj:
    const = tl.load(file_obj)

# Access constants
AGG = const["agg"]
CHRT_BAR = const["chart"]["bar"]
COLORS = const["colors"]
COLS = const["columns"]
DIRECTION = const["train"]["direction"]
SUB_SVC = const["train"]["sub_service"]
TRN = const["train"]

### 1.3 Retrieve performance data

In [11]:
filepath = parent_path.joinpath("data", "processed", "station_performance_metrics-v1p2.csv")
trains = pd.read_csv(
    filepath, dtype={"Address 02": "str", "ZIP Code": "str"}, low_memory=False
)  # avoid DtypeWarning

In [12]:
trains.shape

(68412, 24)

In [13]:
trains.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68412 entries, 0 to 68411
Data columns (total 24 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Fiscal Year                               68412 non-null  int64  
 1   Fiscal Quarter                            68412 non-null  int64  
 2   Service Line                              68412 non-null  object 
 3   Service                                   68412 non-null  object 
 4   Sub Service                               68412 non-null  object 
 5   Route Miles                               68412 non-null  int64  
 6   Train Number                              68412 non-null  int64  
 7   Arrival Station Code                      68412 non-null  object 
 8   Arrival Station                           68412 non-null  object 
 9   Arrival Station Type                      68386 non-null  object 
 10  City                              

In [14]:
trains.head(3)

,Fiscal Year,Fiscal Quarter,Service Line,Service,Sub Service,Route Miles,Train Number,Arrival Station Code,Arrival Station,Arrival Station Type,...,State,Division,Region,Country,Latitude,Longitude,Total Detraining Customers,Late Detraining Customers,Late to Total Detraining Customers Ratio,Late Detraining Customers Avg Min Late
0,2024,3,Long Distance,Auto Train,Auto Train,914,52,LOR,Lorton (Auto Train),Station Building (with waiting room),...,Virginia,South Atlantic,South,United States,38.708143,-77.220942,42445,23316,0.54932,95.0
1,2024,3,Long Distance,Auto Train,Auto Train,914,53,SFA,Sanford (Auto Train),Station Building (with waiting room),...,Florida,South Atlantic,South,United States,28.808544,-81.291274,28034,18439,0.65774,91.0
2,2024,3,Long Distance,California Zephyr,California Zephyr,2408,5,BRL,Burlington,Station Building (with waiting room),...,Iowa,West North Central,Midwest,United States,40.805788,-91.101951,557,223,0.40036,54.0


## 2.0 Select trains: Northeast Corridor (NEC)

Amtrak's Northeast Corridor (NEC) is the busiest passenger rail corridor in the United States. It is
the only Amtrak line that operates high-speed Acela Express service. The NEC is a shared asset with
commuter rail operators, including the Massachusetts Bay Transportation Authority (MBTA),
Metro-North Railroad, New Jersey Transit, Southeastern Pennsylvania Transportation Authority
(SEPTA), and the Maryland Transit Administration (MTA).

### 2.1 _Acela Express_ (Boston - New York - Philadelphia - Washington, D.C.) [1 pt]

Amtrak's [_Acela_](https://www.amtrak.com/acela-train) service is a high-speed rail service(`150`
mph | `240` km/h) that operates along the Northeast Corridor (NEC) between
[South Station](https://www.amtrak.com/stations/bos), Boston, MA
([BOS](https://www.amtrak.com/stations/bos)) and
[Union Station](https://www.amtrak.com/stations/was), Washington, D.C.
([WAS](https://www.amtrak.com/stations/was)). The service features multiple departures daily. The
service features express trains that make limited stops and regional trains that make all stops.

This section focuses on the _Acela Express_ service. Retrieve the _Acela Express_ performance data
by calling the appropriate `amtk_network` function. Assign the return value of the function call to
a variable named `acela_xp`.

In [15]:
# YOUR CODE HERE
acela_xp = ntwk.by_sub_service(trains, SUB_SVC["ace_xp"])
acela_xp.shape

(2453, 24)

In [16]:
# hidden tests are within this cell

### 2.2 _Acela Express_: on-time performance metrics (entire period) [1 pt]

_Acela Express_ performance data is a compilation of quarterly metrics that focus on late
detraining passengers. Detraining assengers are considered on-time if they arrive at their
destination no later than fifteen (`15`) minutes after their scheduled arrival time. All other
detraining passengers are considered late.

In [17]:
# Total train arrivals
acela_xp_trn_arrivals = acela_xp.shape[0]

# Detraining totals
acela_xp_detrn = acela_xp[COLS["total_detrn"]].sum()
acela_xp_detrn_late = acela_xp[COLS["late_detrn"]].sum()
acela_xp_detrn_on_time = acela_xp_detrn - acela_xp_detrn_late

print(
    f"Train Arrivals: {acela_xp_trn_arrivals}",
    f"Total Detraining Customers: {acela_xp_detrn}",
    f"Late Detraining Customers: {acela_xp_detrn_late}",
    f"On-Time Detraining Customers: {acela_xp_detrn_on_time}",
    sep="\n",
)

# Compute summary statistics
acela_xp_stats = detrn.get_sum_stats(acela_xp, AGG["columns"], AGG["funcs"])
acela_xp_stats

Train Arrivals: 2453
Total Detraining Customers: 3310356
Late Detraining Customers: 576649
On-Time Detraining Customers: 2733707


,Train Arrivals,Total Detraining Customers sum,Total Detraining Customers mean,Total Detraining Customers median,Total Detraining Customers std,Late Detraining Customers sum,Late Detraining Customers mean,Late Detraining Customers median,Late Detraining Customers std,Late to Total Detraining Customers Ratio,Late Detraining Customers Avg Min Late mean,Total On Time Detraining Customers sum
0,2453,3310356.0,1349.5132,377.0,2404.6604,576649.0,235.0791,48.0,501.387,0.1742,None,2733707.0


In [18]:
# hidden tests are within this cell

### 2.3 _Acela Express_ trains [1 pt]

Each _Acela Express_ train is identified by a unique train number.

Create a `DataFrame` named `acela_xp_trns` that contains one row for each train comprising the
_Acela Express_ service. Include the following columns in the `DataFrame` in the order specified:

1. "Service Line"
2. "Service"
3. "Sub Service"
4. "Route Miles"
5. "Train Number"

Reset the index (set `drop=True`) when creating the new `DataFrame`.

In [19]:
# YOUR CODE HERE
acela_xp_trns = (
    acela_xp[[COLS["svc_line"], COLS["svc"], COLS["sub_svc"], COLS["route_miles"], COLS["trn"]]]
    .drop_duplicates()
    .sort_values(by=COLS["trn"])   # ← tambahkan sort
    .reset_index(drop=True)
)
acela_xp_trns

,Service Line,Service,Sub Service,Route Miles,Train Number
0,Northeast Corridor,Acela Express,Acela Express,457,2103
1,Northeast Corridor,Acela Express,Acela Express,457,2106
2,Northeast Corridor,Acela Express,Acela Express,457,2121
3,Northeast Corridor,Acela Express,Acela Express,457,2122
4,Northeast Corridor,Acela Express,Acela Express,457,2126
5,Northeast Corridor,Acela Express,Acela Express,457,2128
6,Northeast Corridor,Acela Express,Acela Express,457,2150
7,Northeast Corridor,Acela Express,Acela Express,457,2151
8,Northeast Corridor,Acela Express,Acela Express,457,2152
9,Northeast Corridor,Acela Express,Acela Express,457,2153


In [20]:
# hidden tests are within this cell

### 2.4 _Acela Express_: mean late arrival times summary statistics

Review the central tendency, dispersion, and shape for the mean late arrival times of
_Acela Express_ trains. Call the custom function named `frm.describe_numeric_column()` to return a
dictionary of summary statistics.

In [21]:
# Drop missing values
acela_xp_avg_mm_late = acela_xp[COLS["late_detrn_avg_mm_late"]].dropna().reset_index(drop=True)

# Call the custom frm.describe_numeric_column() function again
acela_xp_avg_mm_late_describe = frm.describe_numeric_column(acela_xp_avg_mm_late)
acela_xp_avg_mm_late_describe

{'type': pandas.core.series.Series,
 'name': 'Late Detraining Customers Avg Min Late',
 'values': {'non_null': np.int64(1986),
  'missing': np.int64(0),
  'dtype': dtype('float64')},
 'center': {'mean': np.float64(30.614300100704934),
  'median': 26.0,
  'mode': np.float64(21.0)},
 'position': {'min': np.float64(11.0),
  '25%': np.float64(20.0),
  '50%': np.float64(26.0),
  '75%': np.float64(35.0),
  'max': np.float64(305.0)},
 'spread': {'variance': 441.80179036631756,
  'std': 21.019081577612223,
  'range': 294.0,
  'iqr': np.float64(15.0)},
 'shape': {'skewness': np.float64(5.309165458832421),
  'kurtosis': np.float64(51.41815789389943)}}

The skewness and kurtosis values returned suggest that the distribution of mean late arrival times
of _Acela Express_ trains are positively skewed and features features a sharper peak and heavier
right tail than a normal distribution. Let's confirm this visually by generating a histogram.

### 2.5 _Acela Express_: visualize distribution of mean late arrival times

Visualize mean late arrival times for the entire period. The data is binned prior to plotting.

#### 2.5.1 Create the chart data

In [22]:
# Convert to DataFrame
acela_xp_avg_mm_late = acela_xp_avg_mm_late.to_frame(name=COLS["avg_mm_late"])

# Get mean and standard deviation
mu = acela_xp_avg_mm_late_describe["center"]["mean"]
sigma = acela_xp_avg_mm_late_describe["spread"]["std"]

# Get max value (for x-axis ticks); pad max value for chart display
max_val = acela_xp_avg_mm_late_describe["position"]["max"]
max_val_ceil = (np.ceil(max_val / 10) * 10).astype(float)

# Create bins
acela_xp_mm_late, bins, num_bins, bin_width = frm.create_bins(
    acela_xp_avg_mm_late, COLS["avg_mm_late"], 10
)

# Bin the data
chrt_data = frm.bin_data(acela_xp_mm_late, COLS["avg_mm_late"], bins)
# chrt_data

#### 2.5.2 Generate the histogram

In [23]:
title = chrt.format_title(
    acela_xp_stats, f"Amtrak {SUB_SVC['ace_xp']} Service Late Detraining Passengers"
)

chart = (
    hst_bld.HistogramBuilder(hst.HistogramChart(frame=chrt_data, title=title))
    .x(
        axis_tick_count=max_val_ceil,
        axis_values=np.arange(0.0, max_val_ceil + 15, 5).tolist(),
        bin_max_bins=num_bins,
        bin_step=bin_width,
        scale_domain=[0.0, max_val_ceil],
    )
    .mu_rule(mu=mu)
    .sigma_rules(sigma=sigma, n=2)
    .build()
)
chart.display()

alt.LayerChart(...)

### 2.6 _Acela Express_, Trains 2155 & 2154

_Acela_ trains 2155 (southbound) and 2154 (northbound) operate daily between
[South Station](https://www.amtrak.com/stations/bos), Boston, MA
([BOS](https://www.amtrak.com/stations/bos)) and
[Union Station](https://www.amtrak.com/stations/was), Washington, D.C. ([WAS](https://www.amtrak.com/stations/was)).

#### 2.6.1 _Acela Express_ Train 2155, southbound, detraining passengers summary statistics [1 pt]

Departing daily from [South Station](https://www.amtrak.com/stations/bos), Boston, MA
([BOS](https://www.amtrak.com/stations/bos)).

In [24]:
# Base columns for routes
rte_cols = [
    COLS["trn"],
    COLS["station_code"],
    COLS["station"],
    COLS["state"],
    COLS["lat"],
    COLS["lon"],
]

# Train 2154 southbound
amtk_2155 = ntwk.by_train_number(trains, 2155)
amtk_2155_rte = ntwk.create_route(amtk_2155, "southbound")
amtk_2155_rte_stats = detrn.get_route_sum_stats(
    amtk_2155_rte,
    COLS["station_code"],
    AGG["columns"],
    AGG["funcs"],
    rte_cols,
)
amtk_2155_rte_stats

,Train Number,Arrival Station Code,Arrival Station,State,Latitude,Longitude,Train Arrivals,Total Detraining Customers sum,Total Detraining Customers mean,Total Detraining Customers median,Total Detraining Customers std,Late Detraining Customers sum,Late Detraining Customers mean,Late Detraining Customers median,Late Detraining Customers std,Late to Total Detraining Customers Ratio,Late Detraining Customers Avg Min Late mean,Total On Time Detraining Customers sum
0,2155,WAS,Washington,District of Columbia,38.896993,-77.006422,12,127575,10631.2500,11164.0,1824.6040,32188,2682.3333,2833.0,1046.0782,0.2523,29.5833,95387
1,2155,BWI,BWI Thurgood Marshall Airport Station,Maryland,39.192362,-76.694300,3,2446,815.3333,839.0,136.0527,691,230.3333,177.0,114.7098,0.2825,30.6667,1755
2,2155,BAL,Baltimore (Penn Station),Maryland,39.307302,-76.615688,12,13187,1098.9167,1100.0,156.4425,3270,272.5000,253.0,110.8164,0.2480,27.6667,9917
3,2155,WIL,Wilmington,Delaware,39.737263,-75.551095,12,9161,763.4167,755.5,117.2088,3191,265.9167,228.5,94.7527,0.3483,24.8333,5970
4,2155,PHL,Philadelphia (Gray 30th St Sta),Pennsylvania,39.955615,-75.181041,12,30483,2540.2500,2516.0,362.5107,8892,741.0000,682.5,289.7519,0.2917,23.1667,21591
5,2155,MET,Metropark (Iselin),New Jersey,40.568056,-74.329583,11,3661,332.8182,387.0,174.7277,957,87.0000,63.0,64.8521,0.2614,22.2000,2704
6,2155,NWK,Newark (Penn Station),New Jersey,40.734706,-74.164750,12,5434,452.8333,480.0,95.5004,1177,98.0833,79.5,68.8285,0.2166,26.1667,4257
7,2155,NYP,NY Moynihan Train Hall at Penn Station,New York,40.751038,-73.996327,12,122032,10169.3333,10580.0,1206.4759,20019,1668.2500,1296.0,968.1477,0.1640,28.2500,102013
8,2155,STM,Stamford,Connecticut,41.047130,-73.542160,12,7912,659.3333,672.5,141.1500,1535,127.9167,99.5,91.3957,0.1940,21.3333,6377
9,2155,NHV,New Haven (Union Station),Connecticut,41.297714,-72.926670,12,4000,333.3333,348.5,85.8342,442,36.8333,16.0,38.7294,0.1105,24.1667,3558


In [25]:
# hidden tests are within this cell

##### 2.6.1.1 Write to file [1 pt]

Write `amtk_2155_rte_stats` to a CSV file named `stu-amtk_2155_rte_stats.csv`. Store the file in the
`data/student` directory. Then compare it to the accompanying `fxt-amtk_2155_rte_stats.csv` file.
It must match line for line, character for character.

In [26]:
# YOUR CODE HERE
filepath = parent_path.joinpath("data", "student", "stu-amtk_2155_rte_stats.csv")
amtk_2155_rte_stats.to_csv(filepath, index=False)

In [27]:
# hidden tests are within this cell

#### 2.6.2 _Acela Express_ Train 2155: late detraining metrics (fiscal year and quarter)

Review the central tendency, dispersion, and shape for the mean late arrival times of
_Acela Express_ trains. Call the custom function named `frm.describe_numeric_column()` to return a
dictionary of summary statistics.


In [29]:
# Drop missing values — gunakan amtk_2155, bukan amtk_2154
amtk_2155_avg_mm_late = amtk_2155[COLS["late_detrn_avg_mm_late"]].dropna().reset_index(drop=True)

# Describe the column
amtk_2155_avg_mm_late_describe = frm.describe_numeric_column(amtk_2155_avg_mm_late)
amtk_2155_avg_mm_late_describe

{'type': pandas.core.series.Series,
 'name': 'Late Detraining Customers Avg Min Late',
 'values': {'non_null': np.int64(121),
  'missing': np.int64(0),
  'dtype': dtype('float64')},
 'center': {'mean': np.float64(25.28099173553719),
  'median': 23.0,
  'mode': np.float64(21.0)},
 'position': {'min': np.float64(12.0),
  '25%': np.float64(20.0),
  '50%': np.float64(23.0),
  '75%': np.float64(29.0),
  'max': np.float64(77.0)},
 'spread': {'variance': 79.07038567493116,
  'std': 8.892153039333678,
  'range': 65.0,
  'iqr': np.float64(9.0)},
 'shape': {'skewness': np.float64(2.2531334382697175),
  'kurtosis': np.float64(9.855369686527695)}}

##### 2.6.2.1 Retrieve the data

In [30]:
# Base columns for average minutes late
cols = [COLS["year"], COLS["quarter"], COLS["late_detrn_avg_mm_late"]]

# Chart data
chrt_data = detrn.get_qtr_avg_min_late(
    amtk_2155_rte, cols, COLS["year_quarter"], [COLORS["amtk_blue"], COLORS["amtk_red"]]
)
chrt_data

,Fiscal Year Quarter,Late Detraining Customers Avg Min Late,Color
0,2021Q4,29.0,#00537e
1,2021Q4,25.0,#00537e
2,2021Q4,28.0,#00537e
3,2021Q4,25.0,#00537e
4,2021Q4,23.0,#00537e
...,...,...,...
136,2024Q3,25.0,#ef3824
137,2024Q3,17.0,#ef3824
138,2024Q3,14.0,#ef3824
139,2024Q3,14.0,#ef3824


##### 2.6.2.2 Preaggregate the data

Attempting to instantiate an instance of a Vega-Altair
[`alt.Chart()`](https://altair-viz.github.io/user_guide/generated/toplevel/altair.Chart.html) class
by passing to it a dataset comprising more than `5000` rows will trigger a `MaxRowsError`. You can
disable the `MaxRows` check by calling `alt.data_transformers.disable_max_rows()` method. However,
disabling the check may result in performance issues, including browser crashes.

The preferred approach when [working with large datasets](https://altair-viz.github.io/user_guide/large_datasets.html#large-datasets) is to _preaggregate_ the data before generating a plot. This can be achieved
"manually"&mdash;the approach adopted in this notebook&mdash;or by
[installing](https://altair-viz.github.io/user_guide/large_datasets.html#installing-vegafusion[) and
[enabling](https://altair-viz.github.io/user_guide/large_datasets.html#enabling-the-vegafusion-data-transformer)
Altair's companion [vegafusion](https://vegafusion.io/) data transformer package.

In [31]:
# Base columns for aggregation statistics
cols = [COLS["year_quarter"], COLS["late_detrn_avg_mm_late"]]

# Pre-aggregate the data
chrt_data = frm.aggregate_data(chrt_data, cols)

##### 2.6.2.3 Generate box plots

Visualize the distribution of mean late arrival times for late detraining passengers. Illustrate with box plots.

In [32]:
# Chart title
title_txt = (
    f"Amtrak {TRN['2155']['name']} Train {TRN['2155']['number']} Late Detraining Passengers\n"
    f"{TRN['2155']['route']} ({TRN['2155']['direction']})"
)
title = chrt.format_title(amtk_2155_rte_stats, title_txt)

chart = (
    bxp_bld.BoxPlotBuilder(bxp.BoxplotPreAggChart(frame=chrt_data, title=title))
    .x(shorthand="Fiscal Year Quarter:N", axis_title="Period")
    .y(shorthand="Late Detraining Customers Avg Min Late:Q", axis_title="Average Minutes Late")
    .build()
)
chart.display()

alt.LayerChart(...)

#### 2.6.3 _Acela Express_ Train 2154, northbound, detraining passengers summary statistics [1 pt]

Departing daily from [Union Station](https://www.amtrak.com/stations/was), Washington, D.C.
([WAS](https://www.amtrak.com/stations/was)).

Review previous code employed to generate summary statistics for an Amtrak train. Then leverage
functions available in the `amtk_network` and `amtk_detrain` modules to create three new
`DataFrame` objects named `amtk_2154`, `amtk_2154_rte`, and `amtk_2154_rte_stats`, respectively.

In [33]:
# YOUR CODE HERE
amtk_2154 = ntwk.by_train_number(trains, 2154)
amtk_2154_rte = ntwk.create_route(amtk_2154, TRN["2154"]["direction"])
amtk_2154_rte_stats = detrn.get_route_sum_stats(
    amtk_2154_rte,
    COLS["station_code"],
    AGG["columns"],
    AGG["funcs"],
    rte_cols,
)
amtk_2154_rte_stats

,Train Number,Arrival Station Code,Arrival Station,State,Latitude,Longitude,Train Arrivals,Total Detraining Customers sum,Total Detraining Customers mean,Total Detraining Customers median,Total Detraining Customers std,Late Detraining Customers sum,Late Detraining Customers mean,Late Detraining Customers median,Late Detraining Customers std,Late to Total Detraining Customers Ratio,Late Detraining Customers Avg Min Late mean,Total On Time Detraining Customers sum
0,2154,BWI,BWI Thurgood Marshall Airport Station,Maryland,39.192362,-76.694300,12,1278,106.5000,95.0,48.6051,41,3.4167,3.0,2.8110,0.0321,27.7000,1237
1,2154,BAL,Baltimore (Penn Station),Maryland,39.307302,-76.615688,12,1669,139.0833,151.5,48.9702,83,6.9167,4.0,6.0221,0.0497,26.2727,1586
2,2154,WIL,Wilmington,Delaware,39.737263,-75.551095,12,4089,340.7500,382.5,129.1955,275,22.9167,22.0,13.8003,0.0673,24.5000,3814
3,2154,PHL,Philadelphia (Gray 30th St Sta),Pennsylvania,39.955615,-75.181041,12,25334,2111.1667,2221.0,518.8538,2468,205.6667,240.5,77.4331,0.0974,25.0000,22866
4,2154,TRE,Trenton,New Jersey,40.219011,-74.754440,1,28,28.0000,28.0,NaN,1,1.0000,1.0,NaN,0.0357,11.0000,27
5,2154,NWK,Newark (Penn Station),New Jersey,40.734706,-74.164750,12,9512,792.6667,842.5,127.6446,1683,140.2500,125.0,71.9319,0.1769,23.0833,7829
6,2154,NYP,NY Moynihan Train Hall at Penn Station,New York,40.751038,-73.996327,12,133093,11091.0833,11278.0,1207.6415,14971,1247.5833,1098.5,634.6931,0.1125,28.0833,118122
7,2154,STM,Stamford,Connecticut,41.047130,-73.542160,12,7586,632.1667,645.0,81.4559,2130,177.5000,146.5,94.4029,0.2808,22.0833,5456
8,2154,NHV,New Haven (Union Station),Connecticut,41.297714,-72.926670,11,7418,674.3636,722.0,162.3387,1596,145.0909,144.0,68.5835,0.2152,25.3636,5822
9,2154,PVD,Providence (Amtrak),Rhode Island,41.829490,-71.413478,12,19647,1637.2500,1668.0,193.5830,6986,582.1667,574.5,224.3264,0.3556,27.0833,12661


In [34]:
# hidden tests are within this cell

##### 2.6.3.1 Write to file [1 pt]

Write `amtk_2154_rte_stats` to a CSV file named `stu-amtk_2154_rte_stats.csv`. Store the file in the
`data/student` directory. Then compare it to the accompanying `fxt-amtk-2154_route_stats.csv` file.
It must match line for line, character for character.

In [35]:
# YOUR CODE HERE
filepath = parent_path.joinpath("data", "student", "stu-amtk_2154_rte_stats.csv")
amtk_2154_rte_stats.to_csv(filepath, index=False)

In [36]:
# hidden tests are within this cell

#### 2.6.4 _Acela Express_ Train 2154: late detraining metrics (fiscal year and quarter)

Review the central tendency, dispersion, and shape for the mean late arrival times of
_Acela Express_ Train 2154. Call the custom function named `frm.describe_numeric_column()` to return
a dictionary of summary statistics. Then visualize each fiscal year and quarter data with a box plot.

In [37]:
# Drop missing values
amtk_2154_avg_mm_late = amtk_2154[COLS["late_detrn_avg_mm_late"]].dropna().reset_index(drop=True)

# Describe the column
amtk_2154_avg_mm_late_describe = frm.describe_numeric_column(amtk_2154_avg_mm_late)
amtk_2154_avg_mm_late_describe

{'type': pandas.core.series.Series,
 'name': 'Late Detraining Customers Avg Min Late',
 'values': {'non_null': np.int64(141),
  'missing': np.int64(0),
  'dtype': dtype('float64')},
 'center': {'mean': np.float64(26.80141843971631),
  'median': 26.0,
  'mode': np.float64(21.0)},
 'position': {'min': np.float64(11.0),
  '25%': np.float64(20.0),
  '50%': np.float64(26.0),
  '75%': np.float64(31.0),
  'max': np.float64(56.0)},
 'spread': {'variance': 83.81742654508611,
  'std': 9.155185773379266,
  'range': 45.0,
  'iqr': np.float64(11.0)},
 'shape': {'skewness': np.float64(0.8783906477596932),
  'kurtosis': np.float64(0.6057990936284594)}}

##### 2.6.4.1 Retrieve the chart data

In [38]:
# Base columns for average minutes late
cols = [COLS["year"], COLS["quarter"], COLS["late_detrn_avg_mm_late"]]

# Chart data
chrt_data = detrn.get_qtr_avg_min_late(
    amtk_2154_rte, cols, COLS["year_quarter"], [COLORS["amtk_blue"], COLORS["amtk_red"]]
)
# chrt_data

##### 2.6.4.2 Preaggregate the data

In [39]:
# Base columns for aggregation statistics
cols = [COLS["year_quarter"], COLS["late_detrn_avg_mm_late"]]

# Pre-aggregate the data
chrt_data = frm.aggregate_data(chrt_data, cols)

##### 2.6.4.3 Generate box plots

In [40]:
# Chart title
title_txt = (
    f"Amtrak {TRN['2154']['name']} Train {TRN['2154']['number']} Late Detraining Passengers\n"
    f"{TRN['2154']['route']} ({TRN['2154']['direction']})"
)
title = chrt.format_title(amtk_2154_rte_stats, title_txt)

chart = (
    bxp_bld.BoxPlotBuilder(bxp.BoxplotPreAggChart(frame=chrt_data, title=title))
    .x(shorthand="Fiscal Year Quarter:N", axis_title="Period")
    .y(shorthand="Late Detraining Customers Avg Min Late:Q", axis_title="Average Minutes Late")
    .build()
)
chart.display()

alt.LayerChart(...)

## 3.0 Select trains: State Supported

Amtrak's state-supported trains are funded by state governments. These services are typically
shorter in length and operate within a single state or across multiple states.

### 3.1 _Pacific Surfliner_ Service (San Luis Obispo - Santa Barbara - Los Angeles - San Diego) [1 pt]

Amtrak's [Pacific Surfliner](https://www.amtrak.com/pacific-surfliner-train) service operates
between San Luis Obispo, CA ([SLO](https://www.amtrak.com/stations/slo)) and
[Santa Fe Depot](https://www.amtrak.com/stations/san), San Diego, CA
([SAN](https://www.amtrak.com/stations/san)). The service features multiple
departures daily and serves a number of popular destinations, including
Santa Barbara, CA ([SBA](https://www.amtrak.com/stations/sba)),
Los Angeles, CA ([LAX](https://www.amtrak.com/stations/lax)), Anaheim, CA
([ANA](https://www.amtrak.com/stations/ana)), and San Juan Capistrano, CA
([SNC](https://www.amtrak.com/stations/snc)).

Retrieve the _Pacific Surfliner_ performance data by calling the appropriate `amtk_network`
function. Assign the return value of the function call to a variable named `surf`.

In [41]:
# YOUR CODE HERE
surf = ntwk.by_sub_service(trains, SUB_SVC["surf"])
surf.shape

(4448, 24)

In [42]:
# hidden tests are within this cell

### 3.2 _Pacific Surfliner_: on-time performance metrics (entire period) [1 pt]

Pacific Surfliner performance data is a compilation of quarterly metrics that focus on late
detraining passengers. Detraining assengers are considered on-time if they arrive at their
destination no later than fifteen (`15`) minutes after their scheduled arrival time. All other
detraining passengers are considered late.

In [43]:
# Total train arrivals
surf_trn_arrivals = surf.shape[0]

# Detraining totals
surf_detrn = surf[COLS["total_detrn"]].sum()
surf_detrn_late = surf[COLS["late_detrn"]].sum()
surf_detrn_on_time = surf_detrn - surf_detrn_late

print(
    f"Train Arrivals: {surf_trn_arrivals}",
    f"Total Detraining Customers: {surf_detrn}",
    f"Late Detraining Customers: {surf_detrn_late}",
    f"On-Time Detraining Customers: {surf_detrn_on_time}",
    sep="\n",
)

# Compute summary statistics
surf_stats = detrn.get_sum_stats(surf, AGG["columns"], AGG["funcs"])
surf_stats

Train Arrivals: 24
Total Detraining Customers: 5040187
Late Detraining Customers: 932657
On-Time Detraining Customers: -4107530


,Train Arrivals,Total Detraining Customers sum,Total Detraining Customers mean,Total Detraining Customers median,Total Detraining Customers std,Late Detraining Customers sum,Late Detraining Customers mean,Late Detraining Customers median,Late Detraining Customers std,Late to Total Detraining Customers Ratio,Late Detraining Customers Avg Min Late mean,Total On Time Detraining Customers sum
0,4448,5040187.0,1133.1356,478.5,1976.3956,932657.0,209.6801,58.0,445.2761,0.185,None,4107530.0


In [44]:
# hidden tests are within this cell

### 3.3 _Pacific Surfliner_ trains [1 pt]

Each _Pacific Surfliner_ train is identified by a unique train number.

Create a `DataFrame` named `surf_trns` that contains one row for each train comprising the
_Pacific Surfliner_ service. Include the following columns in the `DataFrame` in the order specified:

1. "Service Line"
2. "Service"
3. "Sub Service"
4. "Route Miles"
5. "Train Number"

Reset the index (set `drop=True`) when creating the new `DataFrame`.

In [45]:
# YOUR CODE HERE
surf_trns = (
    surf[[COLS["svc_line"], COLS["svc"], COLS["sub_svc"], COLS["route_miles"], COLS["trn"]]]
    .drop_duplicates()
    .reset_index(drop=True)
)
surf_trns

,Service Line,Service,Sub Service,Route Miles,Train Number
0,State Supported,Pacific Surfliner,Pacific Surfliner,351,562
1,State Supported,Pacific Surfliner,Pacific Surfliner,351,564
2,State Supported,Pacific Surfliner,Pacific Surfliner,351,572
3,State Supported,Pacific Surfliner,Pacific Surfliner,351,573
4,State Supported,Pacific Surfliner,Pacific Surfliner,351,580
...,...,...,...,...,...
62,State Supported,Pacific Surfliner,Pacific Surfliner,351,1572
63,State Supported,Pacific Surfliner,Pacific Surfliner,351,1763
64,State Supported,Pacific Surfliner,Pacific Surfliner,351,1768
65,State Supported,Pacific Surfliner,Pacific Surfliner,351,1793


In [46]:
# hidden tests are within this cell

### 3.4 _Pacific Surfliner_: mean late arrival times summary statistics

Review the central tendency, dispersion, and shape for the mean late arrival times of
_Pacific Surfliner_ trains. Call the custom function named `frm.describe_numeric_column()` to return
a dictionary of summary statistics.

In [47]:
# Drop missing values
surf_avg_mm_late = surf[COLS["late_detrn_avg_mm_late"]].dropna().reset_index(drop=True)

# Call the custom frm.describe_numeric_column() function again
surf_avg_mm_late_describe = frm.describe_numeric_column(surf_avg_mm_late)
surf_avg_mm_late_describe

{'type': pandas.core.series.Series,
 'name': 'Late Detraining Customers Avg Min Late',
 'values': {'non_null': np.int64(3774),
  'missing': np.int64(0),
  'dtype': dtype('float64')},
 'center': {'mean': np.float64(42.63222045574987),
  'median': 39.0,
  'mode': np.float64(33.0)},
 'position': {'min': np.float64(2.0),
  '25%': np.float64(31.0),
  '50%': np.float64(39.0),
  '75%': np.float64(49.0),
  'max': np.float64(391.0)},
 'spread': {'variance': 422.04811078520663,
  'std': 20.543809548990826,
  'range': 389.0,
  'iqr': np.float64(18.0)},
 'shape': {'skewness': np.float64(4.752313978802546),
  'kurtosis': np.float64(54.5568427759855)}}

The skewness and kurtosis values returned suggest that the distribution of mean late arrival times
of _Pacific Surfliner_ trains is positively skewed and features a sharper peak and heavier right
tail than a normal distribution. Let's confirm this visually by generating a histogram.

### 3.5 _Pacific Surfliner_: visualize distribution of mean late arrival times

Visualize mean late arrival times for the entire period. The data is binned prior to plotting.

#### 3.5.1 Create the chart data [1 pt]

In [48]:
# Convert to DataFrame
surf_avg_mm_late = surf_avg_mm_late.to_frame(name=COLS["avg_mm_late"])

# Get mean and standard deviation
mu = surf_avg_mm_late_describe["center"]["mean"]
sigma = surf_avg_mm_late_describe["spread"]["std"]

# Get max value (for x-axis ticks); pad max value for chart display
max_val = surf_avg_mm_late_describe["position"]["max"]
max_val_ceil = (np.ceil(max_val / 10) * 10).astype(float)

# Create bins
surf_mm_late, bins, num_bins, bin_width = frm.create_bins(surf_avg_mm_late, COLS["avg_mm_late"], 10)

# Bin the data
chrt_data = frm.bin_data(surf_mm_late, COLS["avg_mm_late"], bins)
# chrt_data

In [49]:
# hidden tests are within this cell

#### 3.5.2 Generate the histogram

In [50]:
title = chrt.format_title(
    surf_stats, f"Amtrak {SUB_SVC['surf']} Service Late Detraining Passengers"
)

chart = (
    hst_bld.HistogramBuilder(hst.HistogramChart(frame=chrt_data, title=title))
    .x(
        axis_tick_count=max_val_ceil,
        axis_values=np.arange(0.0, max_val_ceil + 15, 5).tolist(),
        bin_max_bins=num_bins,
        bin_step=bin_width,
        scale_domain=[0.0, max_val_ceil],
    )
    .mu_rule(mu=mu)
    .sigma_rules(sigma=sigma, n=2)
    .build()
)
chart.display()

alt.LayerChart(...)

### 3.6 _Pacific Surfliner_ Trains 774 & 777

_Pacific Surfliner_ trains 774 (southbound) and 777 (northbound) operate Sunday to Friday between
San Luis Obispo, CA ([SLO](https://www.amtrak.com/stations/slo)) and
[Santa Fe Depot](https://www.amtrak.com/stations/san), San Diego, CA
([SAN](https://www.amtrak.com/stations/san)).

#### 3.6.1 _Pacific Surfliner_ Train 774, southbound, detraining passengers summary statistics [1 pt]

Departing Sunday to Friday from San Luis Obispo, CA ([SLO](https://www.amtrak.com/stations/slo)).

In [51]:
# Base columns for routes
rte_cols = [
    COLS["trn"],
    COLS["station_code"],
    COLS["station"],
    COLS["state"],
    COLS["lat"],
    COLS["lon"],
    ]

# Train 774 southbound
amtk_774 = ntwk.by_train_number(trains, 774)
amtk_774_rte = ntwk.create_route(amtk_774, TRN["774"]["direction"])
amtk_774_rte_stats = detrn.get_route_sum_stats(
    amtk_774_rte,
    COLS["station_code"],
    AGG["columns"],
    AGG["funcs"],
    rte_cols,
)
amtk_774_rte_stats

,Arrival Station Code,Arrival Station,State,Latitude,Longitude,Train Arrivals,Total Detraining Customers sum,Total Detraining Customers mean,Total Detraining Customers median,Total Detraining Customers std,Late Detraining Customers sum,Late Detraining Customers mean,Late Detraining Customers median,Late Detraining Customers std,Late to Total Detraining Customers Ratio,Late Detraining Customers Avg Min Late mean,Total On Time Detraining Customers sum
0,GVB,Grover Beach,California,35.121260,-120.629266,12,146,12.1667,3.5,13.8356,1,0.0833,0.0,0.2887,0.0068,54.0000,145
1,GUA,Guadalupe-Santa Maria,California,34.962927,-120.573410,10,94,9.4000,6.0,11.5297,1,0.1000,0.0,0.3162,0.0106,22.0000,93
2,LPS,Lompoc-Surf,California,34.682704,-120.605001,12,118,9.8333,8.5,4.7832,14,1.1667,1.0,1.1146,0.1186,41.2500,104
3,GTA,Goleta,California,34.437729,-119.843092,12,1729,144.0833,151.0,43.4584,500,41.6667,40.0,21.0641,0.2892,28.3333,1229
4,SBA,Santa Barbara,California,34.413718,-119.692785,12,8608,717.3333,728.5,139.1751,1806,150.5000,140.5,102.3581,0.2098,34.3333,6802
5,CPN,Carpinteria,California,34.396780,-119.522975,12,2929,244.0833,241.5,62.8005,947,78.9167,83.5,50.9393,0.3233,29.0909,1982
6,MPK,Moorpark,California,34.284760,-118.878059,11,1238,112.5455,116.0,47.0773,251,22.8182,17.0,20.2031,0.2027,40.6364,987
7,VEC,Ventura,California,34.276929,-119.299918,12,3773,314.4167,287.5,116.0975,891,74.2500,65.5,50.9637,0.2362,32.5833,2882
8,SIM,Simi Valley,California,34.270204,-118.695163,12,1832,152.6667,147.0,33.9688,423,35.2500,28.0,19.7904,0.2309,45.4167,1409
9,CWT,Chatsworth,California,34.253205,-118.599417,12,4994,416.1667,393.0,112.2836,1213,101.0833,79.5,73.3676,0.2429,40.9167,3781


In [52]:
# hidden tests are within this cell

##### 3.6.1.1 Write to file [1 pt]

Write `amtk_774_rte_stats` to a CSV file named `stu-amtk_774_rte_stats.csv`. Store the file in the
`data/student` directory. Then compare it to the accompanying `fxt-amtk_774_rte_stats.csv` file.
It must match line for line, character for character.

In [53]:
# YOUR CODE HERE
filepath = parent_path.joinpath("data", "student", "stu-amtk_774_rte_stats.csv")
amtk_774_rte_stats.to_csv(filepath, index=False)

In [54]:
# hidden tests are within this cell

#### 3.6.2 _Pacific Surfliner_ Train 774: late detraining metrics (fiscal year and quarter)

Review the central tendency, dispersion, and shape for the mean late arrival times of
_Pacific Surfliner_ Train 774. Call the custom function named `frm.describe_numeric_column()` to
return a dictionary of summary statistics. Then visualize each fiscal year and quarter data with a
box plot.

In [55]:
# Drop missing values
amtk_774_avg_mm_late = amtk_774[COLS["late_detrn_avg_mm_late"]].dropna().reset_index(drop=True)

# Describe the column
amtk_774_avg_mm_late_describe = frm.describe_numeric_column(amtk_774_avg_mm_late)
amtk_774_avg_mm_late_describe

{'type': pandas.core.series.Series,
 'name': 'Late Detraining Customers Avg Min Late',
 'values': {'non_null': np.int64(268),
  'missing': np.int64(0),
  'dtype': dtype('float64')},
 'center': {'mean': np.float64(39.57462686567164),
  'median': 37.0,
  'mode': np.float64(37.0)},
 'position': {'min': np.float64(18.0),
  '25%': np.float64(32.0),
  '50%': np.float64(37.0),
  '75%': np.float64(45.0),
  'max': np.float64(128.0)},
 'spread': {'variance': 147.5112639051932,
  'std': 12.145421520276404,
  'range': 110.0,
  'iqr': np.float64(13.0)},
 'shape': {'skewness': np.float64(2.541218278069484),
  'kurtosis': np.float64(13.800652191508451)}}

##### 3.6.2.1 Retrieve the chart data

In [56]:
# Base columns for average minutes late
cols = [COLS["year"], COLS["quarter"], COLS["late_detrn_avg_mm_late"]]

# Chart data
chrt_data = detrn.get_qtr_avg_min_late(
    amtk_774_rte, cols, COLS["year_quarter"], [COLORS["amtk_blue"], COLORS["amtk_red"]]
)
chrt_data

,Fiscal Year Quarter,Late Detraining Customers Avg Min Late,Color
0,2021Q4,NaN,#00537e
1,2021Q4,22.0,#00537e
2,2021Q4,25.0,#00537e
3,2021Q4,28.0,#00537e
4,2021Q4,30.0,#00537e
...,...,...,...
288,2024Q3,41.0,#ef3824
289,2024Q3,54.0,#ef3824
290,2024Q3,58.0,#ef3824
291,2024Q3,60.0,#ef3824


##### 3.6.2.2 Preaggregate the data

In [57]:
# Base columns for aggregation statistics
cols = [COLS["year_quarter"], COLS["late_detrn_avg_mm_late"]]

# Pre-aggregate the data
chrt_data = frm.aggregate_data(chrt_data, cols)

##### 3.6.2.3 Generate box plots

In [58]:
# Chart title
title_txt = (
    f"Amtrak {TRN['774']['name']} Train {TRN['774']['number']} Late Detraining Passengers\n"
    f"{TRN['774']['route']} ({TRN['774']['direction']})"
)
title = chrt.format_title(amtk_774_rte_stats, title_txt)

chart = (
    bxp_bld.BoxPlotBuilder(bxp.BoxplotPreAggChart(frame=chrt_data, title=title))
    .x(shorthand="Fiscal Year Quarter:N", axis_title="Period")
    .y(shorthand="Late Detraining Customers Avg Min Late:Q", axis_title="Average Minutes Late")
    .build()
)
chart.display()

alt.LayerChart(...)

#### 3.6.3 _Pacific Surfliner_ Train 777, westbound, detraining passengers summary statistics [1 pt]

Departs Sunday to Friday from [Santa Fe Depot](https://www.amtrak.com/stations/san), San Diego, CA
([SAN](https://www.amtrak.com/stations/san)).

Review previous code employed to generate summary statistics for an Amtrak train. Then leverage
functions available in the `amtk_network` and `amtk_detrain` modules to create three new
`DataFrame` objects named `amtk_777`, `amtk_777_rte`, and `amtk_777_rte_stats`, respectively.

In [59]:
amtk_777 = ntwk.by_train_number(trains, 777)
amtk_777_rte = ntwk.create_route(amtk_777, TRN["777"]["direction"])
amtk_777_rte_stats = detrn.get_route_sum_stats(
    amtk_777_rte,
    COLS["station_code"],   # ← hapus COLS["trn"] yang extra
    AGG["columns"],
    AGG["funcs"],
    rte_cols,
)
amtk_777_rte_stats

,Arrival Station Code,Arrival Station,State,Latitude,Longitude,Train Arrivals,Total Detraining Customers sum,Total Detraining Customers mean,Total Detraining Customers median,Total Detraining Customers std,Late Detraining Customers sum,Late Detraining Customers mean,Late Detraining Customers median,Late Detraining Customers std,Late to Total Detraining Customers Ratio,Late Detraining Customers Avg Min Late mean,Total On Time Detraining Customers sum
0,OLT,San Diego (Old Town),California,32.755266,-117.200073,11,344,31.2727,37.0,19.9253,5,0.4545,0.0,0.8202,0.0145,29.6667,339
1,SOL,Solana Beach,California,32.992937,-117.271135,11,2227,202.4545,185.0,143.0569,23,2.0909,1.0,2.9139,0.0103,40.1667,2204
2,OSD,Oceanside,California,33.192515,-117.379430,11,4740,430.9091,456.0,163.0751,138,12.5455,12.0,11.8858,0.0291,48.5556,4602
3,SNC,San Juan Capistrano,California,33.501318,-117.663813,11,11642,1058.3636,1186.0,437.4074,900,81.8182,62.0,69.2717,0.0773,53.4000,10742
4,IRV,Irvine,California,33.656771,-117.733695,12,14275,1189.5833,1382.5,584.7832,1080,90.0000,78.5,70.7377,0.0757,55.4545,13195
5,SNA,Santa Ana,California,33.751629,-117.856607,12,9223,768.5833,819.5,272.3101,760,63.3333,51.0,44.7667,0.0824,51.0000,8463
6,ANA,Anaheim,California,33.803803,-117.877308,12,20234,1686.1667,1513.5,550.7651,1781,148.4167,130.5,80.1322,0.0880,51.5833,18453
7,FUL,Fullerton,California,33.868969,-117.922849,12,17801,1483.4167,1488.0,214.2112,1801,150.0833,150.0,66.3057,0.1012,44.5833,16000
8,LAX,Los Angeles,California,34.056177,-118.236778,12,128371,10697.5833,10209.0,1638.2428,11042,920.1667,828.0,324.0058,0.0860,56.0833,117329
9,GDL,Glendale,California,34.123706,-118.258868,12,9433,786.0833,724.0,263.4117,1161,96.7500,72.0,112.2636,0.1231,57.5833,8272


In [60]:
# hidden tests are within this cell

##### 3.6.3.1 Write to file [1 pt]

Write `amtk_777_rte_stats` to a CSV file named `stu-amtk_777_rte_stats.csv`. Store the file in the
`data/student` directory. Then compare it to the accompanying `fxt-amtk_777_rte_stats.csv` file.
It must match line for line, character for character.

In [61]:
# YOUR CODE HERE
filepath = parent_path.joinpath("data", "student", "stu-amtk_777_rte_stats.csv")
amtk_777_rte_stats.to_csv(filepath, index=False)

In [62]:
# hidden tests are within this cell

#### 3.6.4 _Pacific Surfliner_ Train 777: late detraining metrics (fiscal year and quarter) [1 pt]

Review the central tendency, dispersion, and shape for the mean late arrival times of
_Pacific Surfliner_ Train 777. Call the custom function named `frm.describe_numeric_column()` to
return a dictionary of summary statistics. Then visualize each fiscal year and quarter data with a
box plot.

In [63]:
# Drop missing values
amtk_777_avg_mm_late = amtk_777[COLS["late_detrn_avg_mm_late"]].dropna().reset_index(drop=True)  # ← tambahkan .dropna()

# Describe the column
amtk_777_avg_mm_late_describe = frm.describe_numeric_column(amtk_777_avg_mm_late)
amtk_777_avg_mm_late_describe

{'type': pandas.core.series.Series,
 'name': 'Late Detraining Customers Avg Min Late',
 'values': {'non_null': np.int64(296),
  'missing': np.int64(17),
  'dtype': dtype('float64')},
 'center': {'mean': np.float64(49.12837837837838),
  'median': 45.0,
  'mode': np.float64(40.0)},
 'position': {'min': np.float64(17.0),
  '25%': np.float64(37.0),
  '50%': np.float64(45.0),
  '75%': np.float64(56.25),
  'max': np.float64(137.0)},
 'spread': {'variance': 349.14956481905637,
  'std': 18.68554427409211,
  'range': 120.0,
  'iqr': np.float64(nan)},
 'shape': {'skewness': np.float64(1.7720953121744818),
  'kurtosis': np.float64(4.790974152163351)}}

In [64]:
# hidden tests are within this cell

##### 3.6.4.1 Retrieve the chart data

In [65]:
# Base columns for average minutes late
cols = [COLS["year"], COLS["quarter"], COLS["late_detrn_avg_mm_late"]]

# Chart data
chrt_data = detrn.get_qtr_avg_min_late(
    amtk_777_rte, cols, COLS["year_quarter"], [COLORS["amtk_blue"], COLORS["amtk_red"]]
)
# chrt_data

##### 3.6.4.2 Preaggregate the data

In [66]:
# Base columns for aggregation statistics
cols = [COLS["year_quarter"], COLS["late_detrn_avg_mm_late"]]

# Pre-aggregate the data
chrt_data = frm.aggregate_data(chrt_data, cols)

##### 3.6.4.3 Generate box plots

Visualize the distribution of mean late arrival times for late detraining passengers. Illustrate with box plots.

In [67]:
# Chart title
title_txt = (
    f"Amtrak {TRN['777']['name']} Train {TRN['777']['number']} Late Detraining Passengers\n"
    f"{TRN['777']['route']} ({TRN['777']['direction']})"
)
title = chrt.format_title(amtk_777_rte_stats, title_txt)

chart = (
    bxp_bld.BoxPlotBuilder(bxp.BoxplotPreAggChart(frame=chrt_data, title=title))
    .x(shorthand="Fiscal Year Quarter:N", axis_title="Period")
    .y(shorthand="Late Detraining Customers Avg Min Late:Q", axis_title="Average Minutes Late")
    .build()
)
chart.display()

alt.LayerChart(...)

## 4.0 Long-Distance trains

Amtrak's long-distance trains operate across the United States. These trains are typically overnight
services that connect major cities and regions. Long-distance trains are known for their scenic
routes and dining car services.

### 4.1 _City of New Orleans_ service (Chicago - Memphis - New Orleans) [1 pt]

The [_City of New Orleans_](https://www.amtrak.com/city-of-new-orleans-train), operates daily
between [Chicago Union Station](https://www.amtrak.com/stations/chi), Chicago, IL
([CHI](https://www.amtrak.com/stations/chi)) and
[Union Passenger Terminal](https://www.amtrak.com/stations/NOL), New Orleans, LA
([NOL](https://www.amtrak.com/stations/NOL)) via
[Central Station](https://www.amtrak.com/stations/mem), Memphis, TN
([MEM](https://www.amtrak.com/stations/mem)). The train revives the name of the Illinois Central
Railroad's
[_City of New Orleans_](https://en.wikipedia.org/wiki/City_of_New_Orleans_(train)) that operated
between 1947 and 1971. The train is also known for its association with the classic tune
["City of New Orleans"](https://en.wikipedia.org/wiki/City_of_New_Orleans_(song)) written by Steve
Goodman and popularized by Arlo Guthrie.

Retrieve the _City of New Orleans_ performance data by calling the appropriate `amtk_network`
function. Assign the return value of the function call to a variable named `cno`.

In [89]:
# YOUR CODE HERE
cno = ntwk.by_sub_service(trains, SUB_SVC["cno"])
cno.shape

(536, 24)

In [90]:
# hidden tests are within this cell

### 4.2 _City of New Orleans_: on-time performance metrics (entire period) [1 pt]

_City of New Orleans_ performance data is a compilation of quarterly metrics that focus on late
detraining passengers. Detraining assengers are considered on-time if they arrive at their
destination no later than fifteen (`15`) minutes after their scheduled arrival time. All other
detraining passengers are considered late.

In [91]:
# Total train arrivals
cno_trn_arrivals = cno.shape[0]

# Detraining totals
cno_detrn = cno[COLS["total_detrn"]].sum()
cno_detrn_late = cno[COLS["late_detrn"]].sum()
cno_detrn_on_time = cno_detrn - cno_detrn_late

print(
    f"Train Arrivals: {cno_trn_arrivals}",
    f"Total Detraining Customers: {cno_detrn}",
    f"Late Detraining Customers: {cno_detrn_late}",
    f"On-Time Detraining Customers: {cno_detrn_on_time}",
    sep="\n",
)

# Compute summary statistics
cno_stats = detrn.get_sum_stats(cno, AGG["columns"], AGG["funcs"])
cno_stats

Train Arrivals: 536
Total Detraining Customers: 570194
Late Detraining Customers: 159831
On-Time Detraining Customers: 410363


,Train Arrivals,Total Detraining Customers sum,Total Detraining Customers mean,Total Detraining Customers median,Total Detraining Customers std,Late Detraining Customers sum,Late Detraining Customers mean,Late Detraining Customers median,Late Detraining Customers std,Late to Total Detraining Customers Ratio,Late Detraining Customers Avg Min Late mean,Total On Time Detraining Customers sum
0,536,570194.0,1063.7948,212.5,2503.7947,159831.0,298.1922,95.5,605.9606,0.2803,None,410363.0


In [92]:
# hidden tests are within this cell

### 4.3 _City of New Orleans_ trains [1 pt]

Each _City of New Orleans_ train is identified by a unique train number.

Create a `DataFrame` named `cno_trns` that contains one row for each train comprising the
_City of New Orleans_ service. Include the following columns in the `DataFrame` in the order specified:

1. "Service Line"
2. "Service"
3. "Sub Service"
4. "Route Miles"
5. "Train Number"

Reset the index (set `drop=True`) when creating the new `DataFrame`.

In [93]:
# YOUR CODE HERE
cno_trns = (
    cno[[COLS["svc_line"], COLS["svc"], COLS["sub_svc"], COLS["route_miles"], COLS["trn"]]]
    .drop_duplicates()
    .reset_index(drop=True)
)
cno_trns

,Service Line,Service,Sub Service,Route Miles,Train Number
0,Long Distance,City Of New Orleans,City Of New Orleans,930,58
1,Long Distance,City Of New Orleans,City Of New Orleans,930,59
2,Long Distance,City Of New Orleans,City Of New Orleans,930,1058
3,Long Distance,City Of New Orleans,City Of New Orleans,930,1059


In [94]:
# hidden tests are within this cell

### 4.4 _City of New Orleans_: mean late arrival times summary statistics

Review the central tendency, dispersion, and shape for the mean late arrival times of
_City of New Orleans_ trains. Call the custom function named `frm.describe_numeric_column()` to
return a dictionary of summary statistics.

In [95]:
# Drop missing values
cno_avg_mm_late = cno[COLS["late_detrn_avg_mm_late"]].dropna().reset_index(drop=True)

# Call the custom frm.describe_numeric_column() function again
cno_avg_mm_late_describe = frm.describe_numeric_column(cno_avg_mm_late)
cno_avg_mm_late_describe

{'type': pandas.core.series.Series,
 'name': 'Late Detraining Customers Avg Min Late',
 'values': {'non_null': np.int64(493),
  'missing': np.int64(0),
  'dtype': dtype('float64')},
 'center': {'mean': np.float64(71.50101419878297),
  'median': 68.0,
  'mode': np.float64(68.0)},
 'position': {'min': np.float64(18.0),
  '25%': np.float64(51.0),
  '50%': np.float64(68.0),
  '75%': np.float64(85.0),
  'max': np.float64(387.0)},
 'spread': {'variance': 1251.2423770180908,
  'std': 35.372904560102086,
  'range': 369.0,
  'iqr': np.float64(34.0)},
 'shape': {'skewness': np.float64(2.54874272793014),
  'kurtosis': np.float64(15.515159904674796)}}

The skewness and kurtosis values returned suggest that the distribution of mean late arrival times
of _City of New Orleans_ trains is positively skewed and features a sharper peak and heavier right
tail than a normal distribution. Let's confirm this visually by generating a histogram.

### 4.5 _City of New Orleans_: visualize distribution of mean late arrival times

Visualize mean late arrival times for the entire period. The data is binned prior to plotting.

#### 4.5.1 Create the chart data

In [97]:
print("cno shape:", cno.shape)
print("SUB_SVC['cno']:", SUB_SVC["cno"])
print("Unique sub_svc di trains:", trains[COLS["sub_svc"]].unique())
print("cno avg mm late non-null:", cno[COLS["late_detrn_avg_mm_late"]].notna().sum())

cno shape: (536, 24)
SUB_SVC['cno']: City of New Orleans
Unique sub_svc di trains: ['Auto Train' 'California Zephyr' 'Capitol Ltd' 'Cardinal'
 'City Of New Orleans' 'Coast Starlight' 'Crescent' 'Empire Builder'
 'Lake Shore Ltd' 'Palmetto' 'Silver Meteor' 'Silver Star'
 'Southwest Chief' 'Sunset Ltd' 'Texas Eagle' 'Acela'
 'On Spine Northeast Regional' 'Richmond / Newport News / Norfolk'
 'Roanoke' 'Springfield Shuttles' 'Borealis' 'Capitol Corridor'
 'Carolinian' 'Cascades' 'Downeaster' 'Adirondack' 'Berkshire Flyer'
 'Ethan Allen Express' 'Maple Leaf' 'New York - Albany'
 'New York - Niagara Falls' 'Heartland Flyer' 'Hiawatha'
 'Carl Sandburg / Illinois Zephyr' 'Illini / Saluki' 'Lincoln Service'
 'Keystone' 'Lincoln / Missouri' 'Blue Water' 'Pere Marquette' 'Wolverine'
 'Missouri' 'Pacific Surfliner' 'Pennsylvanian' 'Piedmont' 'San Joaquins'
 'Vermonter' 'Acela Express']
cno avg mm late non-null: 493


In [98]:
# Fix by_sub_service agar case-insensitive
def by_sub_service(frame, sub_service):
    mask = frame[COLS["sub_svc"]].str.lower() == sub_service.lower()
    return frame[mask].reset_index(drop=True)

ntwk.by_sub_service = by_sub_service
print("✓ by_sub_service patch diperbaiki!")

✓ by_sub_service patch diperbaiki!


In [99]:
# Convert to DataFrame
cno_avg_mm_late = cno_avg_mm_late.to_frame(name=COLS["avg_mm_late"])

# Get mean and standard deviation
mu = cno_avg_mm_late_describe["center"]["mean"]
sigma = cno_avg_mm_late_describe["spread"]["std"]  # ← tambahkan ["std"]

# Get max value (for x-axis ticks); pad max value for chart display
max_val = cno_avg_mm_late_describe["position"]["max"]  # ← hapus .astype(int)
max_val_ceil = (np.ceil(max_val / 10) * 10).astype(float)

# Create bins
cno_mm_late, bins, num_bins, bin_width = frm.create_bins(cno_avg_mm_late, COLS["avg_mm_late"], 10)

# Bin the data
chrt_data = frm.bin_data(cno_mm_late, COLS["avg_mm_late"], bins)

#### 4.5.2 Generate the histogram

In [100]:
title = chrt.format_title(cno_stats, f"Amtrak {SUB_SVC['cno']} Service Late Detraining Passengers")

chart = (
    hst_bld.HistogramBuilder(hst.HistogramChart(frame=chrt_data, title=title))
    .x(
        axis_tick_count=max_val_ceil,
        axis_values=np.arange(0.0, max_val_ceil + 15, 5).tolist(),
        bin_max_bins=num_bins,
        bin_step=bin_width,
        scale_domain=[0.0, max_val_ceil],
    )
    .mu_rule(mu=mu)
    .sigma_rules(sigma=sigma, n=2)
    .build()
)
chart.display()

alt.LayerChart(...)

### 4.6 _City of New Orleans_, Train 59 and 58

City of New Orleans trains 59 (southbound) and 58 (northbound) operate daily between
[Chicago Union Station](https://www.amtrak.com/stations/chi), Chicago, IL
([CHI](https://www.amtrak.com/stations/chi)) and
[Union Passenger Terminal](https://www.amtrak.com/stations/NOL), New Orleans, LA
([NOL](https://www.amtrak.com/stations/NOL)).

#### 4.6.1 _City of New Orleans_ Train 59, southbound, detraining passengers summary statistics [1 pt]

Departs daily from [Chicago Union Station](https://www.amtrak.com/stations/chi), Chicago, IL
([CHI](https://www.amtrak.com/stations/chi)).

In [101]:
# Base columns for routes
rte_cols = [
    COLS["trn"],  
    COLS["station_code"],
    COLS["station"],
    COLS["state"],
    COLS["lat"],
    COLS["lon"],
]

# Train 59 southbound
amtk_59 = ntwk.by_train_number(trains, 59)   # ← 58 → 59
amtk_59_rte = ntwk.create_route(amtk_59, TRN["59"]["direction"])
amtk_59_rte_stats = detrn.get_route_sum_stats(
    amtk_59_rte, COLS["station_code"], AGG["columns"], AGG["funcs"], rte_cols,
)
amtk_59_rte_stats

,Arrival Station Code,Arrival Station,State,Latitude,Longitude,Train Arrivals,Total Detraining Customers sum,Total Detraining Customers mean,Total Detraining Customers median,Total Detraining Customers std,Late Detraining Customers sum,Late Detraining Customers mean,Late Detraining Customers median,Late Detraining Customers std,Late to Total Detraining Customers Ratio,Late Detraining Customers Avg Min Late mean,Total On Time Detraining Customers sum
0,CHI,Chicago (Union Station),Illinois,41.878992,-87.641015,11,150011,13637.3636,13166.0,2608.6811,34699,3154.4545,3033.0,1418.1027,0.2313,95.2727,115312
1,HMW,Homewood,Illinois,41.562446,-87.668685,11,15746,1431.4545,1305.0,409.8503,8037,730.6364,650.0,212.4577,0.5104,78.4545,7709
2,KKI,Kankakee,Illinois,41.119259,-87.865430,11,2736,248.7273,247.0,62.6324,1275,115.9091,112.0,35.9790,0.4660,75.7273,1461
3,CHM,Champaign-Urbana,Illinois,40.115843,-88.241366,11,11355,1032.2727,1043.0,201.8505,4950,450.0000,394.0,124.2578,0.4359,81.8182,6405
4,MAT,Mattoon,Illinois,39.482730,-88.376045,11,1363,123.9091,137.0,28.8252,565,51.3636,49.0,15.3445,0.4145,83.0000,798
5,EFG,Effingham,Illinois,39.117059,-88.547110,11,1117,101.5455,98.0,27.8186,480,43.6364,40.0,16.2989,0.4297,86.6364,637
6,CEN,Centralia,Illinois,38.527531,-89.136118,11,1014,92.1818,91.0,23.1768,360,32.7273,32.0,12.2889,0.3550,80.9091,654
7,CDL,Carbondale (Amtrak),Illinois,37.724235,-89.216628,11,6995,635.9091,669.0,146.1687,2151,195.5455,170.0,60.2534,0.3075,83.9091,4844
8,FTN,Fulton,Kentucky,36.525704,-88.888772,11,2237,203.3636,200.0,73.6441,867,78.8182,80.0,32.5786,0.3876,76.0909,1370
9,NBN,Newbern-Dyersburg,Tennessee,36.112711,-89.262264,11,1459,132.6364,145.0,55.9665,639,58.0909,50.0,28.1370,0.4380,62.0909,820


In [102]:
# hidden tests are within this cell

##### 4.6.1.1 Write to file [1 pt]

Write `amtk_59_rte_stats` to a CSV file named `stu-amtk_59_rte_stats.csv`. Store the file in the
`data/student` directory. Then compare it to the accompanying `fxt-amtk-59_route_stats.csv` file.
It must match line for line, character for character.

In [103]:
# YOUR CODE HERE
filepath = parent_path.joinpath("data", "student", "stu-amtk_59_rte_stats.csv")
amtk_59_rte_stats.to_csv(filepath, index=False)

In [104]:
# hidden tests are within this cell

#### 4.6.2 _City of New Orleans_ Train 59: late detraining metrics (fiscal year and quarter)

Review the central tendency, dispersion, and shape for the mean late arrival times of
_City of New Orleans_ Train 59. Call the custom function named `frm.describe_numeric_column()` to
return a dictionary of summary statistics. Then visualize each fiscal year and quarter data with a
box plot.

In [105]:
# Drop missing values
amtk_59_avg_mm_late = amtk_59[COLS["late_detrn_avg_mm_late"]].dropna().reset_index(drop=True)

# Describe the column
amtk_59_avg_mm_late_describe = frm.describe_numeric_column(amtk_59_avg_mm_late)
amtk_59_avg_mm_late_describe

{'type': pandas.core.series.Series,
 'name': 'Late Detraining Customers Avg Min Late',
 'values': {'non_null': np.int64(209),
  'missing': np.int64(0),
  'dtype': dtype('float64')},
 'center': {'mean': np.float64(66.76555023923444),
  'median': 62.0,
  'mode': np.float64(76.0)},
 'position': {'min': np.float64(23.0),
  '25%': np.float64(45.0),
  '50%': np.float64(62.0),
  '75%': np.float64(81.0),
  'max': np.float64(221.0)},
 'spread': {'variance': 960.7764998159735,
  'std': 30.996394948702882,
  'range': 198.0,
  'iqr': np.float64(36.0)},
 'shape': {'skewness': np.float64(1.4509054944484152),
  'kurtosis': np.float64(3.869952017175466)}}

##### 4.6.2.1 Retrieve the chart data

In [106]:
# Base columns for chart data
cols = [COLS["year"], COLS["quarter"], COLS["late_detrn_avg_mm_late"]]

# Get the chart data
chrt_data = detrn.get_qtr_avg_min_late(
    amtk_59_rte, cols, COLS["year_quarter"], [COLORS["amtk_blue"], COLORS["amtk_red"]]
)
chrt_data

,Fiscal Year Quarter,Late Detraining Customers Avg Min Late,Color
0,2022Q1,96.0,#ef3824
1,2022Q1,74.0,#ef3824
2,2022Q1,59.0,#ef3824
3,2022Q1,73.0,#ef3824
4,2022Q1,58.0,#ef3824
...,...,...,...
204,2024Q3,45.0,#ef3824
205,2024Q3,39.0,#ef3824
206,2024Q3,37.0,#ef3824
207,2024Q3,36.0,#ef3824


##### 4.6.2.2 Preaggregate the data

In [107]:
# Base columns for aggregation statistics
cols = [COLS["year_quarter"], COLS["late_detrn_avg_mm_late"]]

# Pre-aggregate the data
chrt_data = frm.aggregate_data(chrt_data, cols)

##### 4.6.2.3 Generate box plots

In [108]:
# Chart title
title_txt = (
    f"Amtrak {TRN['59']['name']} Train {TRN['59']['number']} Late Detraining Passengers\n"
    f"{TRN['59']['route']} ({TRN['59']['direction']})"
)
title = chrt.format_title(amtk_59_rte_stats, title_txt)

chart = (
    bxp_bld.BoxPlotBuilder(bxp.BoxplotPreAggChart(frame=chrt_data, title=title))
    .x(shorthand="Fiscal Year Quarter:N", axis_title="Period")
    .y(shorthand="Late Detraining Customers Avg Min Late:Q", axis_title="Average Minutes Late")
    .build()
)
chart.display()

alt.LayerChart(...)

#### 4.6.3 _City of New Orleans_ Train 58, northbound, detraining passengers summary statistics [1 pt]

Departs daily from [Union Passenger Terminal](https://www.amtrak.com/stations/NOL), New Orleans, LA
([NOL](https://www.amtrak.com/stations/NOL)).

Review previous code employed to generate summary statistics for an Amtrak train. Then leverage
functions available in the `amtk_network` and `amtk_detrain` modules to create three new
`DataFrame` objects named `amtk_58`, `amtk_58_rte`, and `amtk_58_rte_stats`, respectively.

In [109]:
# YOUR CODE HERE
# Base columns for routes
rte_cols = [
    COLS["trn"],  
    COLS["station_code"],
    COLS["station"],
    COLS["state"],
    COLS["lat"],
    COLS["lon"],
]

amtk_58 = ntwk.by_train_number(trains, 58)
amtk_58_rte = ntwk.create_route(amtk_58, TRN["58"]["direction"])
amtk_58_rte_stats = detrn.get_route_sum_stats(
    amtk_58_rte,
    COLS["station_code"],
    AGG["columns"],
    AGG["funcs"],
    rte_cols,
)
amtk_58_rte_stats

,Arrival Station Code,Arrival Station,State,Latitude,Longitude,Train Arrivals,Total Detraining Customers sum,Total Detraining Customers mean,Total Detraining Customers median,Total Detraining Customers std,Late Detraining Customers sum,Late Detraining Customers mean,Late Detraining Customers median,Late Detraining Customers std,Late to Total Detraining Customers Ratio,Late Detraining Customers Avg Min Late mean,Total On Time Detraining Customers sum
0,HMD,Hammond,Louisiana,30.507180,-90.462169,11,2264,205.8182,198.0,69.5698,1126,102.3636,98.0,40.1952,0.4973,30.6364,1138
1,MCB,McComb,Mississippi,31.244467,-90.451334,11,1698,154.3636,155.0,48.5042,1025,93.1818,89.0,30.8539,0.6037,34.4545,673
2,BRH,Brookhaven,Mississippi,31.582961,-90.441070,11,1558,141.6364,150.0,60.3892,985,89.5455,78.0,39.7577,0.6322,41.9091,573
3,HAZ,Hazlehurst,Mississippi,31.861320,-90.394347,11,959,87.1818,78.0,31.1731,657,59.7273,55.0,22.4058,0.6851,38.3636,302
4,JAN,Jackson,Mississippi,32.300808,-90.190936,11,21050,1913.6364,1970.0,573.9058,2584,234.9091,203.0,163.0825,0.1228,63.2727,18466
5,YAZ,Yazoo City,Mississippi,32.848477,-90.415230,11,993,90.2727,72.0,50.3212,200,18.1818,17.0,10.9619,0.2014,60.8182,793
6,GWD,Greenwood,Mississippi,33.517159,-90.176454,11,4154,377.6364,379.0,131.9661,1155,105.0000,96.0,48.0687,0.2780,61.6364,2999
7,MKS,Marks,Mississippi,34.258227,-90.272342,11,2664,242.1818,208.0,69.0924,1112,101.0909,106.0,42.7819,0.4174,59.3636,1552
8,MEM,Memphis,Tennessee,35.132458,-90.059109,11,43340,3940.0000,4042.0,1226.0761,11764,1069.4545,870.0,538.3936,0.2714,74.1818,31576
9,NBN,Newbern-Dyersburg,Tennessee,36.112711,-89.262264,11,1459,132.6364,145.0,55.9665,639,58.0909,50.0,28.1370,0.4380,62.0909,820


In [110]:
# hidden tests are within this cell

##### 4.6.3.1 Write to file [1 pt]

Write `amtk_58_rte_stats` to a CSV file named `stu-amtk_58_rte_stats.csv`. Store the file in the
`data/student` directory. Then compare it to the accompanying `fxt-amtk-58_route_stats.csv` file.
It must match line for line, character for character.

In [111]:
# YOUR CODE HERE
filepath = parent_path.joinpath("data", "student", "stu-amtk_58_rte_stats.csv")
amtk_58_rte_stats.to_csv(filepath, index=False)

In [112]:
# hidden tests are within this cell

#### 4.6.4 _City of New Orleans_ Train 58: late detraining metrics (fiscal year and quarter)

Review the central tendency, dispersion, and shape for the mean late arrival times of
_City of New Orleans_ Train 58. Call the custom function named `frm.describe_numeric_column()` to
return a dictionary of summary statistics. Then visualize each fiscal year and quarter data with a
box plot.

In [113]:
# Drop missing values
amtk_58_avg_mm_late = amtk_58[COLS["late_detrn_avg_mm_late"]].dropna().reset_index(drop=True)

# Describe the column
amtk_58_avg_mm_late_describe = frm.describe_numeric_column(amtk_58_avg_mm_late)
amtk_58_avg_mm_late_describe

{'type': pandas.core.series.Series,
 'name': 'Late Detraining Customers Avg Min Late',
 'values': {'non_null': np.int64(209),
  'missing': np.int64(0),
  'dtype': dtype('float64')},
 'center': {'mean': np.float64(66.76555023923444),
  'median': 62.0,
  'mode': np.float64(76.0)},
 'position': {'min': np.float64(23.0),
  '25%': np.float64(45.0),
  '50%': np.float64(62.0),
  '75%': np.float64(81.0),
  'max': np.float64(221.0)},
 'spread': {'variance': 960.7764998159735,
  'std': 30.996394948702882,
  'range': 198.0,
  'iqr': np.float64(36.0)},
 'shape': {'skewness': np.float64(1.4509054944484152),
  'kurtosis': np.float64(3.869952017175466)}}

##### 4.6.4.1 Retrieve the chart data [1 pt]

In [114]:
# Base columns for average minutes late
cols = [COLS["year"], COLS["quarter"], COLS["late_detrn_avg_mm_late"]]

# Chart data
chrt_data = detrn.get_qtr_avg_min_late(
    amtk_58_rte, cols, COLS["year_quarter"], [COLORS["amtk_blue"], COLORS["amtk_red"]]
)
chrt_data

,Fiscal Year Quarter,Late Detraining Customers Avg Min Late,Color
0,2022Q1,96.0,#ef3824
1,2022Q1,74.0,#ef3824
2,2022Q1,59.0,#ef3824
3,2022Q1,73.0,#ef3824
4,2022Q1,58.0,#ef3824
...,...,...,...
204,2024Q3,45.0,#ef3824
205,2024Q3,39.0,#ef3824
206,2024Q3,37.0,#ef3824
207,2024Q3,36.0,#ef3824


In [115]:
# hidden tests are within this cell

##### 4.6.4.2 Preaggregate the data

In [117]:
# Base columns for average minutes late
cols = [COLS["year_quarter"], COLS["late_detrn_avg_mm_late"]]  # ← avg_min → avg_mm

# Pre-aggregate the data
chrt_data = frm.aggregate_data(chrt_data, cols)

##### 4.6.4.3 Generate box plots

In [118]:
# Chart title
title_txt = (
    f"Amtrak {TRN['58']['name']} Train {TRN['58']['number']} Late Detraining Passengers\n"
    f"{TRN['58']['route']} ({TRN['58']['direction']})"
)
title = chrt.format_title(amtk_58_rte_stats, title_txt)

chart = (
    bxp_bld.BoxPlotBuilder(bxp.BoxplotPreAggChart(frame=chrt_data, title=title))
    .x(shorthand="Fiscal Year Quarter:N", axis_title="Period")
    .y(shorthand="Late Detraining Customers Avg Min Late:Q", axis_title="Average Minutes Late")
    .build()
)
chart.display()

alt.LayerChart(...)

## 5.0 Watermark

In [119]:
%load_ext watermark
%watermark -h -i -iv -m -v

Python implementation: CPython
Python version       : 3.11.9
IPython version      : 8.26.0

Compiler    : GCC 12.3.0
OS          : Linux
Release     : 6.5.0-1024-aws
Machine     : x86_64
Processor   : x86_64
CPU cores   : 32
Architecture: 64bit

Hostname: e3ef964e1464

pandas: 2.2.3
scipy : 1.13.0
numpy : 2.1.3



In [121]:
import pathlib as pl
parent_path = pl.Path.cwd()

for name in ["amtk_2155_rte_stats", "amtk_2154_rte_stats", "amtk_774_rte_stats", 
             "amtk_777_rte_stats", "amtk_59_rte_stats", "amtk_58_rte_stats"]:
    stu = parent_path.joinpath("data", "student", f"stu-{name}.csv")
    # cari file fxt - nama filenya berbeda
    import glob
    fxt_files = glob.glob(str(parent_path.joinpath("data", "student", f"fxt-*{name.split('_rte')[0]}*")))
    print(f"STU: {stu.name}")
    print(f"FXT candidates: {[f.split('/')[-1] for f in fxt_files]}")
    if stu.exists():
        with open(stu) as f:
            print("STU line 1:", repr(f.readlines()[1]))
    if fxt_files:
        with open(fxt_files[0]) as f:
            print("FXT line 1:", repr(f.readlines()[1]))
    print()

STU: stu-amtk_2155_rte_stats.csv
FXT candidates: ['fxt-amtk_2155_rte_stats.csv']
STU line 1: '0,2155,WAS,Washington,District of Columbia,38.896993,-77.006422,12,127575,10631.25,11164.0,1824.604,32188,2682.3333,2833.0,1046.0782,0.2523,29.5833,95387\n'
FXT line 1: '2155,BBY,Boston (Back Bay Station),Massachusetts,42.347317,-71.075828,9,17,1.8889,2.0,0.928,0,0.0,0.0,0.0,0.0,,17\n'

STU: stu-amtk_2154_rte_stats.csv
FXT candidates: ['fxt-amtk_2154_rte_stats.csv']
STU line 1: '0,2154,BWI,BWI Thurgood Marshall Airport Station,Maryland,39.192362,-76.6943,12,1278,106.5,95.0,48.6051,41,3.4167,3.0,2.811,0.0321,27.7,1237\n'
FXT line 1: '2154,BWI,BWI Thurgood Marshall Airport Station,Maryland,39.192362,-76.6943,12,1278,106.5,95.0,48.6051,41,3.4167,3.0,2.811,0.0321,27.7,1237\n'

STU: stu-amtk_774_rte_stats.csv
FXT candidates: ['fxt-amtk_774_rte_stats.csv']
STU line 1: '0,GVB,Grover Beach,California,35.12126,-120.629266,12,146,12.1667,3.5,13.8356,1,0.0833,0.0,0.2887,0.0068,54.0,145\n'
FXT line 1: '77